**Creating external location for silver**

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS motor_silver_location
URL 'abfss://silver@motorlakehrk.dfs.core.windows.net'
WITH (STORAGE CREDENTIAL motor_adls_credential)


Creating Unity Catalog schema for silver

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS motordatabricks.silver

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# ============================================================
# 1. BRONZE PATH
# ============================================================

bronze_path = "abfss://bronze@motorlakehrk.dfs.core.windows.net/"


# ============================================================
# 2. SILVER PATH
# ============================================================

silver_path = "abfss://silver@motorlakehrk.dfs.core.windows.net/motor_telemetry/"


# ============================================================
# 3. CHECKPOINT PATH
# ============================================================

checkpoint_path = "abfss://silver@motorlakehrk.dfs.core.windows.net/checkpoints/motor_telemetry/"


# ============================================================
# 4. BRONZE SCHEMA
# ============================================================

bronze_schema = StructType([
    StructField("Body", StringType(), True),
    StructField("EnqueuedTimeUtc", StringType(), True)
])


# ============================================================
# 5. READ BRONZE AS STREAM
# ============================================================

df_bronze = (
    spark.readStream
    .format("json")
    .schema(bronze_schema)
    .option("recursiveFileLookup", "true")
    .load(bronze_path)
)


# ============================================================
# 6. MOTOR JSON SCHEMA INSIDE BODY
# ============================================================

motor_schema = StructType([
    StructField("motor_id", StringType(), True),
    StructField("rpm", IntegerType(), True),
    StructField("temperature", DoubleType(), True),
    StructField("timestamp", StringType(), True),
    StructField("vibration", DoubleType(), True),
    StructField("current", DoubleType(), True)
])


# ============================================================
# 7. PARSE BODY JSON
# ============================================================

df_silver = (
    df_bronze
    .withColumn(
        "motor",
        from_json(col("Body"), motor_schema)
    )
    .select(
        col("motor.motor_id").alias("motor_id"),
        col("motor.rpm").alias("rpm"),
        col("motor.temperature").alias("temperature"),
        col("motor.vibration").alias("vibration"),
        col("motor.current").alias("current"),
        to_timestamp(
            col("motor.timestamp")
        ).alias("event_timestamp"),
        to_timestamp(
            col("EnqueuedTimeUtc")
        ).alias("enqueued_time_utc")
    )
)


# ============================================================
# 8. DATA QUALITY / CLEANING
# ============================================================

df_silver_clean = (
    df_silver
    .filter(col("motor_id").isNotNull())
    .filter(col("rpm").isNotNull())
    .filter(col("temperature").isNotNull())
    .filter(col("event_timestamp").isNotNull())
)


# ============================================================
# 9. WRITE STREAM TO SILVER
# ============================================================

silver_query = (
    df_silver_clean
    .writeStream
    .format("delta")
    .outputMode("append")
    .trigger(availableNow=True)
    .option("checkpointLocation", checkpoint_path)
    .start(silver_path)
)


# ============================================================
# 10. WAIT FOR COMPLETION
# ============================================================

silver_query.awaitTermination()


# The above code works if we are having cluster and use .trigger(processingTime="10 seconds") but we are using lakeflow because

**Lakeflow + Serverless = managed, production-style streaming pipeline with less infrastructure management.
For our IoT project, the main benefits are:**

- Serverless → no cluster creation/configuration.
- Lakeflow → manages streaming, checkpoints, retries, dependencies, and pipeline orchestration.
- Auto Loader → efficiently detects new Bronze files.
- Cost → you pay for the serverless pipeline compute used; it isn't automatically cheaper in every workload, but it avoids keeping your own cluster running.

In [0]:
%sql
SHOW SCHEMAS IN motordatabricks

databaseName
default
information_schema
silver


In [0]:
%sql
SHOW GRANTS ON SCHEMA motordatabricks.silver;

Principal,ActionType,ObjectType,ObjectKey


In [0]:
%sql
GRANT USE CATALOG
ON CATALOG motordatabricks
TO `1833017mdcs@cit.edu.in`;

GRANT USE SCHEMA
ON SCHEMA motordatabricks.silver
TO `1833017mdcs@cit.edu.in`;

GRANT CREATE TABLE
ON SCHEMA motordatabricks.silver
TO `1833017mdcs@cit.edu.in`;

In [0]:
%sql
SHOW GRANTS ON SCHEMA motordatabricks.silver;

Principal,ActionType,ObjectType,ObjectKey
1833017mdcs@cit.edu.in,CREATE TABLE,SCHEMA,motordatabricks.silver
1833017mdcs@cit.edu.in,USE SCHEMA,SCHEMA,motordatabricks.silver


In [0]:
%sql
SHOW GRANTS ON CATALOG motordatabricks;

Principal,ActionType,ObjectType,ObjectKey
1833017mdcs@cit.edu.in,USE CATALOG,CATALOG,motordatabricks
_workspace_users_motordatabricks_7405615735018931,USE CATALOG,CATALOG,motordatabricks


In [0]:
%sql
SELECT *
FROM motordatabricks.silver.motor_telemetry
LIMIT 20;

motor_id,rpm,temperature,vibration,current,event_timestamp,enqueued_time_utc
MTR001,1659,78.17,2.85,14.25,2026-08-19T06:52:58.043Z,2026-08-19T06:52:58.459Z
MTR004,1684,83.71,3.18,14.17,2026-08-19T06:52:58.455Z,2026-08-19T06:52:58.881Z
MTR001,1530,69.39,2.76,11.83,2026-08-19T06:53:08.599Z,2026-08-19T06:53:09.008Z
MTR004,1590,69.9,4.86,14.32,2026-08-19T06:53:08.987Z,2026-08-19T06:53:09.399Z
MTR002,1542,68.68,7.45,8.38,2026-08-19T06:52:58.185Z,2026-08-19T06:52:58.593Z
MTR003,1567,67.29,1.93,8.13,2026-08-19T06:52:58.323Z,2026-08-19T06:52:58.734Z
MTR002,1611,83.31,2.14,11.35,2026-08-19T06:53:08.735Z,2026-08-19T06:53:09.141Z
MTR003,1640,76.81,5.33,10.32,2026-08-19T06:53:08.863Z,2026-08-19T06:53:09.266Z
MTR001,1407,61.21,5.13,11.96,2026-08-19T06:32:53.865Z,2026-08-19T06:32:54.281Z
MTR004,1513,89.57,3.25,14.38,2026-08-19T06:32:54.257Z,2026-08-19T06:32:54.671Z


In [0]:
%sql
DESCRIBE TABLE motordatabricks.silver.motor_telemetry;

col_name,data_type,comment
motor_id,string,null
rpm,int,null
temperature,double,null
vibration,double,null
current,double,null
event_timestamp,timestamp,null
enqueued_time_utc,timestamp,null


In [0]:
%sql
DESCRIBE DETAIL motordatabricks.silver.motor_telemetry;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,0e3d945e-e84b-4dbd-9437-18ccb9b02930,motordatabricks.silver.__materialization_mat_1f9e0630_9dbf_45ae_a69c_b10bc296e5d8_motor_telemetry_1,null,abfss://unity-catalog-storage@dbstoragegly57pmv6m5ru.dfs.core.windows.net/7405615735018931/__unitystorage/catalogs/a295ac3e-c954-4bbc-9db5-d165b4d65841/tables/51edba02-8e8a-45cf-b6b8-9f991358f5be,2026-08-20T04:19:53.085Z,2026-08-20T04:20:06.000Z,List(),List(),1,14946,"Map(delta.parquet.compression.codec -> zstd, spark.internal.pipelines.reconciliation_query_without_timetravel -> SELECT * FROM `motordatabricks`.`silver`.`__materialization_mat_1f9e0630_9dbf_45ae_a69c_b10bc296e5d8_motor_telemetry_1`, pipeline_internal.catalogType -> UNITY_CATALOG, delta.enableChangeDataFeed -> true, spark.sql.internal.pipelines.parentTableId -> 49166df6-38f5-4c8b-9cbf-2827aedb6b2e, delta.enableDeletionVectors -> true, pipelines.pipelineId -> 1f9e0630-9dbf-45ae-a69c-b10bc296e5d8, pipeline_internal.enzymeMode -> Advanced, spark.internal.streaming_table.parentTable -> `motordatabricks`.`silver`.`motor_telemetry`, delta.enableRowTracking -> true, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-71b65fbc-32a7-476e-993c-d91abd6e2fb0, pipelines.metastore.tableName -> `motordatabricks`.`silver`.`__materialization_mat_1f9e0630_9dbf_45ae_a69c_b10bc296e5d8_motor_telemetry_1`, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-f8060540-49d8-4215-994a-0931e80bffa3)",3,7,"List(appendOnly, changeDataFeed, deletionVectors, domainMetadata, invariants, rowTracking)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
DROP TABLE IF EXISTS motordatabricks.silver.motor_telemetry;

In [0]:
%sql
SHOW EXTERNAL LOCATIONS;

name,url,comment
motor_bronze_location,abfss://bronze@motorlakehrk.dfs.core.windows.net/,null
motor_silver_location,abfss://silver@motorlakehrk.dfs.core.windows.net/,null
motordatabricks,abfss://unity-catalog-storage@dbstoragegly57pmv6m5ru.dfs.core.windows.net/7405615735018931,null


In [0]:
%sql
DROP TABLE IF EXISTS motordatabricks.silver.motor_telemetry;

In [0]:
%sql
SHOW GRANTS ON EXTERNAL LOCATION motor_silver_location;

Principal,ActionType,ObjectType,ObjectKey


In [0]:
%sql
GRANT READ FILES, WRITE FILES, CREATE EXTERNAL TABLE
ON EXTERNAL LOCATION motor_silver_location
TO `1833017mdcs@cit.edu.in`;

In [0]:
%sql
SHOW GRANTS ON EXTERNAL LOCATION motor_silver_location;

Principal,ActionType,ObjectType,ObjectKey
1833017mdcs@cit.edu.in,CREATE EXTERNAL TABLE,EXTERNAL LOCATION,motor_silver_location
1833017mdcs@cit.edu.in,READ FILES,EXTERNAL LOCATION,motor_silver_location
1833017mdcs@cit.edu.in,WRITE FILES,EXTERNAL LOCATION,motor_silver_location


In [0]:
%sql
CREATE TABLE motordatabricks.silver.motor_telemetry
USING DELTA
LOCATION 'abfss://silver@motorlakehrk.dfs.core.windows.net/motor_telemetry/';

In [0]:
%sql
DESCRIBE DETAIL motordatabricks.silver.motor_telemetry;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,1b01de08-1a71-4c4b-bbd1-16dd18708f3c,motordatabricks.silver.motor_telemetry,null,abfss://silver@motorlakehrk.dfs.core.windows.net/motor_telemetry,2026-08-20T04:51:40.737Z,2026-08-20T04:51:41.000Z,List(),List(),0,0,Map(delta.enableDeletionVectors -> true),3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
DESCRIBE TABLE motordatabricks.silver.motor_telemetry;

col_name,data_type,comment


In [0]:
%sql
DROP TABLE IF EXISTS motordatabricks.silver.motor_telemetry;

In [0]:
%sql
CREATE TABLE motordatabricks.silver.motor_telemetry (
    motor_id STRING,
    rpm INT,
    temperature DOUBLE,
    vibration DOUBLE,
    current DOUBLE,
    event_timestamp TIMESTAMP,
    enqueued_time_utc TIMESTAMP
)
USING DELTA
LOCATION 'abfss://silver@motorlakehrk.dfs.core.windows.net/motor_telemetry/';

In [0]:
%sql
SELECT
    motor_id,
    rpm,
    temperature,
    vibration,
    current,
    event_timestamp
FROM motordatabricks.silver.motor_telemetry
ORDER BY event_timestamp DESC;

motor_id,rpm,temperature,vibration,current,event_timestamp
MTR004,1590,69.9,4.86,14.32,2026-08-19T06:53:08.987Z
MTR003,1640,76.81,5.33,10.32,2026-08-19T06:53:08.863Z
MTR002,1611,83.31,2.14,11.35,2026-08-19T06:53:08.735Z
MTR001,1530,69.39,2.76,11.83,2026-08-19T06:53:08.599Z
MTR004,1684,83.71,3.18,14.17,2026-08-19T06:52:58.455Z
MTR003,1567,67.29,1.93,8.13,2026-08-19T06:52:58.323Z
MTR002,1542,68.68,7.45,8.38,2026-08-19T06:52:58.185Z
MTR001,1659,78.17,2.85,14.25,2026-08-19T06:52:58.043Z
MTR004,1644,80.16,2.23,9.99,2026-08-19T06:52:47.905Z
MTR003,1694,81.63,6.99,11.49,2026-08-19T06:52:47.766Z
